# Data Cleaning -- Test Dataset
**This notebook prepares Flatiron Health CSV files for patients with advanced head and neck cancer in anticipation of training a gradient-boosted survival model to predict mortality from time of first-line treatment. Specifically, it processes and cleans the test cohort.**

## Import packages

In [1]:
import numpy as np
import pandas as pd

from flatiron_cleaner import DataProcessorHeadNeck, merge_dataframes

## Clean CSV files 

In [2]:
test_ids = pd.read_csv('../outputs/test_patient_ids.csv')

In [3]:
test_ids = test_ids.PatientID.to_list()

In [4]:
index_date_df = pd.read_csv('../data/LineOfTherapy.csv')

In [5]:
index_date_df = (
    index_date_df
    .query('LineNumber == 1')
    .query('IsMaintenanceTherapy == False')
    [['PatientID', 'StartDate']]
)

In [6]:
index_date_df.shape

(8746, 2)

In [7]:
index_date_df = (
    index_date_df[index_date_df.PatientID.isin(test_ids)]
    [['PatientID', 'StartDate']]
)

In [8]:
index_date_df.shape

(1750, 2)

In [9]:
processor = DataProcessorHeadNeck()

### Process Enhanced_AdvHeadNeck.csv

In [10]:
enhanced_df = processor.process_enhanced(file_path = '../data/Enhanced_AdvHeadNeck.csv',
                                         index_date_df = index_date_df,
                                         index_date_column = 'StartDate',
                                         drop_dates = False)

2025-11-28 21:45:53,665 - INFO - Successfully read Enhanced_AdvHeadNeck.csv file with shape: (11347, 15) and unique PatientIDs: 11347
2025-11-28 21:45:53,669 - INFO - Successfully filtered Enhanced_AdvHeadNeck.csv file with shape: (1750, 16) and unique PatientIDs: 1750
2025-11-28 21:45:53,679 - INFO - Successfully processed Enhanced_AdvHeadNeck.csv file with final shape: (1750, 20) and unique PatientIDs: 1750


In [11]:
enhanced_df['days_adv_to_treatment'] = (enhanced_df['imported_StartDate'] - enhanced_df['AdvancedDiagnosisDate']).dt.days
enhanced_df['days_adv_to_treatment'] = np.where(enhanced_df['days_adv_to_treatment'] < 0 , 0, enhanced_df['days_adv_to_treatment'])

In [12]:
enhanced_df['days_diagnosis_to_adv'] = np.where(enhanced_df['days_diagnosis_to_adv'] < 0 , 0, enhanced_df['days_diagnosis_to_adv'])
enhanced_df['days_diagnosis_to_adv'] = enhanced_df['days_diagnosis_to_adv'].fillna(0)

In [13]:
enhanced_df['GroupStage_mod'] = enhanced_df["GroupStage_mod"].map({
    '0': 0, 
    'I': 1,
    'II': 2,
    'III': 3,
    'IV_NOS': 4,
    'IV_locoregional': 4,
    'IV_metastatic': 5, 
    'unknown': np.nan
})

enhanced_df['GroupStage_mod_na'] = np.where(enhanced_df['GroupStage_mod'].isna(), 1, 0)

# impute 4 since stage IV (locoregional) is most common
enhanced_df['GroupStage_mod'] = enhanced_df['GroupStage_mod'].fillna(4)

In [14]:
# impute unknown as smokers since most common
enhanced_df['SmokingStatus'] = (
    enhanced_df['SmokingStatus']
    .map({'History of smoking': 1, 
          'No history of smoking': 0,
          'Unknown/not documented': 1})
    .astype('Int64')
)

In [15]:
enhanced_df['HPVStatus_mod'] = enhanced_df['HPVStatus_mod'].fillna('unknown') 

In [16]:
enhanced_df = enhanced_df.drop(columns = ['DiagnosisDate', 
                                          'AdvancedDiagnosisDate', 
                                          'imported_StartDate',
                                          'FirstLocalRecurDate',
                                          'FirstDistantRecurDate',
                                          'PrimarySurgeryDate',
                                          'PrimaryRadiationDate'])

### Process Demographics.csv 

In [17]:
demographics_df = processor.process_demographics(file_path = '../data/Demographics.csv',
                                                 index_date_df = index_date_df,
                                                 index_date_column = 'StartDate')

2025-11-28 21:45:53,707 - INFO - Successfully read Demographics.csv file with shape: (11347, 6) and unique PatientIDs: 11347
2025-11-28 21:45:53,715 - INFO - Successfully processed Demographics.csv file with final shape: (1750, 6) and unique PatientIDs: 1750


In [18]:
# Impute missing with male
demographics_df['sex_male'] = np.where(demographics_df['Gender'] == 'F', 0, 1)

In [19]:
demographics_df = demographics_df.drop(columns = ['Gender'])

### Process Enhanced_AdvNSCLCBiomarkers.csv

In [20]:
biomarkers_df = processor.process_biomarkers(file_path = '../data/Enhanced_AdvHeadNeckBiomarkers.csv',
                                             index_date_df = index_date_df, 
                                             index_date_column = 'StartDate',
                                             days_before = None, 
                                             days_after = 30)

2025-11-28 21:45:53,731 - INFO - Successfully read Enhanced_AdvHeadNeckBiomarkers.csv file with shape: (4379, 17) and unique PatientIDs: 3131
2025-11-28 21:45:53,736 - INFO - Successfully merged Enhanced_AdvHeadNeckBiomarkers.csv df with index_date_df resulting in shape: (742, 18) and unique PatientIDs: 537
2025-11-28 21:45:53,755 - INFO - Successfully processed Enhanced_AdvHeadNeckBiomarkers.csv file with final shape: (1750, 3) and unique PatientIDs: 1750


In [21]:
biomarkers_df['PDL1_status'] = biomarkers_df['PDL1_status'].fillna('unknown')

### Process ECOG.csv

In [22]:
ecog_df = processor.process_ecog(file_path = '../data/ECOG.csv', 
                                 index_date_df = index_date_df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 days_before_further = 180)

2025-11-28 21:45:53,838 - INFO - Successfully read ECOG.csv file with shape: (199312, 4) and unique PatientIDs: 8957
2025-11-28 21:45:53,868 - INFO - Successfully merged ECOG.csv df with index_date_df resulting in shape: (36959, 5) and unique PatientIDs: 1471
2025-11-28 21:45:53,892 - INFO - Successfully processed ECOG.csv file with final shape: (1750, 3) and unique PatientIDs: 1750


In [23]:
ecog_df['ecog_index'] = ecog_df['ecog_index'].astype('float64')
ecog_df['ecog_index_na'] = np.where(ecog_df['ecog_index'].isna(), 1, 0)

# impute 1 for missing ECOG since most common
ecog_df['ecog_index'] = ecog_df['ecog_index'].fillna(1)

In [24]:
ecog_df['ecog_newly_gte2'] = ecog_df['ecog_newly_gte2'].fillna(0)

### Process Vitals.csv

In [25]:
vitals_df = processor.process_vitals(file_path = '../data/Vitals.csv',
                                     index_date_df = index_date_df,
                                     index_date_column = 'StartDate',
                                     weight_days_before = 90,
                                     days_after = 0,
                                     vital_summary_lookback = 180, 
                                     abnormal_reading_threshold = 1)

2025-11-28 21:45:57,566 - INFO - Successfully read Vitals.csv file with shape: (4024923, 16) and unique PatientIDs: 11339
2025-11-28 21:45:58,930 - INFO - Successfully merged Vitals.csv df with index_date_df resulting in shape: (698763, 17) and unique PatientIDs: 1750
2025-11-28 21:45:59,216 - INFO - Successfully processed Vitals.csv file with final shape: (1750, 8) and unique PatientIDs: 1750


### Process Lab.csv

In [26]:
labs_df = processor.process_labs(file_path = '../data/Lab.csv',
                                 index_date_df = index_date_df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 summary_lookback = 180)

2025-11-28 21:46:11,029 - INFO - Successfully read Lab.csv file with shape: (9064402, 17) and unique PatientIDs: 10989
2025-11-28 21:46:12,861 - INFO - Successfully merged Lab.csv df with index_date_df resulting in shape: (1641752, 18) and unique PatientIDs: 1731
2025-11-28 21:46:16,207 - INFO - Successfully processed Lab.csv file with final shape: (1750, 76) and unique PatientIDs: 1750


### Process MedicationAdministration.csv

In [27]:
medications_df = processor.process_medications(file_path = '../data/MedicationAdministration.csv',
                                               index_date_df = index_date_df,
                                               index_date_column = 'StartDate',
                                               days_before = 90,
                                               days_after = 0)

2025-11-28 21:46:17,579 - INFO - Successfully read MedicationAdministration.csv file with shape: (1098540, 11) and unique PatientIDs: 9978
2025-11-28 21:46:17,837 - INFO - Successfully merged MedicationAdministration.csv df with index_date_df resulting in shape: (188491, 12) and unique PatientIDs: 1713
2025-11-28 21:46:17,868 - INFO - Successfully processed MedicationAdministration.csv file with final shape: (1750, 9) and unique PatientIDs: 1750


### Process Diagnosis.csv

In [28]:
diagnosis_df = processor.process_diagnosis(file_path = '../data/Diagnosis.csv',
                                           index_date_df = index_date_df,
                                           index_date_column = 'StartDate',
                                           days_before = None,
                                           days_after = 0)

2025-11-28 21:46:18,317 - INFO - Successfully read Diagnosis.csv file with shape: (696197, 6) and unique PatientIDs: 11347
2025-11-28 21:46:18,397 - INFO - Successfully merged Diagnosis.csv df with index_date_df resulting in shape: (116468, 7) and unique PatientIDs: 1750
2025-11-28 21:46:18,688 - INFO - Successfully processed Diagnosis.csv file with final shape: (1750, 38) and unique PatientIDs: 1750


### Process Enhanced_Mortality_V2.csv

In [29]:
mortality_df = processor.process_mortality(file_path = '../data/Enhanced_Mortality_V2.csv',
                                           index_date_df = index_date_df, 
                                           index_date_column = 'StartDate',
                                           visit_path = '../data/Visit.csv', 
                                           telemedicine_path = '../data/Telemedicine.csv',
                                           drop_dates = False)

2025-11-28 21:46:18,702 - INFO - Successfully read Enhanced_Mortality_V2.csv file with shape: (7857, 2) and unique PatientIDs: 7857
2025-11-28 21:46:18,709 - INFO - Successfully merged Enhanced_Mortality_V2.csv df with index_date_df resulting in shape: (1750, 3) and unique PatientIDs: 1750
2025-11-28 21:46:19,059 - INFO - The following columns ['last_visit_date'] are used to calculate the last EHR date
2025-11-28 21:46:19,062 - INFO - Successfully processed Enhanced_Mortality_V2.csv file with final shape: (1750, 6) and unique PatientIDs: 1750. There are 0 out of 1750 patients with missing duration values


In [30]:
mortality_df = mortality_df[['PatientID', 'event', 'duration']]

In [31]:
mortality_df = mortality_df.query('duration >= 0')

## Merge dataframes

In [32]:
testing_df = merge_dataframes(enhanced_df,
                              demographics_df,
                              biomarkers_df,
                              ecog_df,
                              vitals_df,
                              labs_df,
                              medications_df,
                              diagnosis_df, 
                              mortality_df,
                              merge_type = 'inner')

2025-11-28 21:46:19,080 - INFO - Anticipated number of merges: 8
2025-11-28 21:46:19,081 - INFO - Anticipated number of columns in final dataframe presuming all columns are unique except for PatientID: 154
2025-11-28 21:46:19,081 - INFO - Dataset 1 shape: (1750, 15), unique PatientIDs: 1750
2025-11-28 21:46:19,082 - INFO - Dataset 2 shape: (1750, 6), unique PatientIDs: 1750
2025-11-28 21:46:19,082 - INFO - Dataset 3 shape: (1750, 3), unique PatientIDs: 1750
2025-11-28 21:46:19,082 - INFO - Dataset 4 shape: (1750, 4), unique PatientIDs: 1750
2025-11-28 21:46:19,083 - INFO - Dataset 5 shape: (1750, 8), unique PatientIDs: 1750
2025-11-28 21:46:19,084 - INFO - Dataset 6 shape: (1750, 76), unique PatientIDs: 1750
2025-11-28 21:46:19,084 - INFO - Dataset 7 shape: (1750, 9), unique PatientIDs: 1750
2025-11-28 21:46:19,085 - INFO - Dataset 8 shape: (1750, 38), unique PatientIDs: 1750
2025-11-28 21:46:19,085 - INFO - Dataset 9 shape: (1744, 3), unique PatientIDs: 1744
2025-11-28 21:46:19,088 - 

In [33]:
testing_df.shape

(1744, 154)

## Export dataframe

In [34]:
testing_df.to_csv('../outputs/1L_features_testing.csv', index = False)

In [35]:
# Save dtypes
testing_df.dtypes.apply(lambda x: x.name).to_csv('../outputs/1L_features_testing_dtypes.csv')